# K-Nearest Neighbor Lab
Read over the sklearn info on [nearest neighbor learners](https://scikit-learn.org/stable/modules/neighbors.html#nearest-neighbors-classification)




In [ ]:
from sklearn.metrics.pairwise import distance_metrics
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
import numpy as np
import pandas as pd
from scipy.io import arff
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1 K-Nearest Neighbor (KNN) algorithm

### 1.1 (15%) Basic KNN Classification

Learn the [Glass data set](https://archive.ics.uci.edu/dataset/42/glass+identification) using [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier) with default parameters.
- Randomly split your data into train/test.  Anytime we don't tell you specifics (such as what percentage is train vs test) choose your own reasonable values
- Give typical train and test set accuracies after running with different random splits
- Print the output probabilities for a test set (predict_proba)
- Try it with different p values (Minkowskian exponent) and discuss any differences

In [ ]:
glass = pd.read_csv("glass.data")
glass.describe()

In [ ]:
# Learn the glass data
X = glass.iloc[:, :-1]  # All columns except last
y = glass.iloc[:, -1]
print("Dataset shape:", X.shape)
print("Number of classes:", y.nunique())
print("Class distribution:\n", y.value_counts().sort_index())
print("\n" + "="*60 + "\n")
n_runs = 5
train_scores = []
test_scores = []
for i in range(n_runs):
    # Random 75/25 train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=i
    )
    knn = KNeighborsClassifier()
    knn.fit(X_train, y_train)

# Get probability predictions for first 5 test samples
proba = knn.predict_proba(X_test[:5])
predictions = knn.predict(X_test[:5])

print("Output Probabilities for First 5 Test Samples:")
print(f"Classes: {knn.classes_}")
print("\nSample | Predicted Class | Probabilities")
for i in range(5):
    proba_str = " ".join([f"{p:.3f}" for p in proba[i]])
    print(f"  {i+1}    |       {predictions[i]}         | [{proba_str}]")
    print(f"       | Actual: {y_test.iloc[i]}      |")

p_values = [1, 2, 3, 5, 10]
results = []

print("Different p Values:")
for p in p_values:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42
    )

    knn = KNeighborsClassifier(p=p)
    knn.fit(X_train, y_train)

    train_acc = knn.score(X_train, y_train)
    test_acc = knn.score(X_test, y_test)

    results.append({
        'p': p,
        'distance': 'Manhattan' if p == 1 else ('Euclidean' if p == 2 else f'Minkowski p={p}'),
        'train_acc': train_acc,
        'test_acc': test_acc
    })

    print(f"p = {p:2d} ({results[-1]['distance']:20s}): "
          f"Train = {train_acc:.4f}, Test = {test_acc:.4f}")


#### Discussion
What were your accuracies or output probabilities and how did different hyperparameter values affect the outcome? Discuss the differences you see.

For the testing and training, we had high accuraacies for all of the different values that we used and the different parameters. I decided to use p values of 3, 5 and 10 for Minkowski. I think it is interesting that the test accuracies were the same when using Minkowskki values of 5 and 10. I also found it interesting that when we used a higher p-value, our accuacy lowered.

## 2 KNN Classification with normalization and distance weighting

Use the [magic telescope](https://axon.cs.byu.edu/data/uci_class/MagicTelescope.arff) dataset

### 2.1 (5%) - Without Normalization or Distance Weighting
- Do random 80/20 train/test splits each time
- Run with k=3 and *without* distance weighting and *without* normalization
- Show train and test set accuracy

In [ ]:
telescope = pd.read_csv("telescope_data.csv")
telescope.head()


In [ ]:
knn = KNeighborsClassifier(n_neighbors=3, weights='uniform')
X = telescope.drop('class', axis=1)
y = telescope['class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"Test Accuracy: {knn.score(X_test, y_test)}")
print(f"Train Accuracy:{knn.score(X_train, y_train)}")
print(f"Difference in Accuracy: {knn.score(X_test, y_test) - knn.score(X_train, y_train)}")

#### Discussion
What did you observe in your results?

I noticed that our test accuracy is just slightly lower than our training accuracy. This in many ways makes sense because the telescope dataset is relatively small. There is also a high chance of overfitting because the test and accuracies are both 99%.

### 2.2 (10%) With Normalization
- Try it with k=3 without distance weighting but *with* normalization of input features.  You may use any reasonable normalization approach (e.g. standard min-max normalization between 0-1, z-transform, etc.)

In [ ]:
# Train/Predict with normalization
from sklearn.preprocessing import StandardScaler
X = telescope.drop('class', axis=1)
y = telescope['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=3, weights='uniform')
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"Test Accuracy: {knn.score(X_test, y_test)}")
print(f"Train Accuracy:{knn.score(X_train, y_train)}")
print(f"Difference in Accuracy: {knn.score(X_test, y_test) - knn.score(X_train, y_train)}")

#### Discussion
Discuss the results of using normalized data vs. unnormalized data

When we normalized the data, we saw a decrease in test and train accuracy but a smaller difference between training and testing. This could be evidence that there was overfitting and that we started to minimize that overfitting, we didn't get rid of it but by normalizing the data, we minimized it at least a little.

### 2.3 (10%) With Distance Weighting
- Try it with k=3 and with distance weighting *and* normalization

In [ ]:
#Train/Precdict with normalization and distance weighting
X = telescope.drop('class', axis=1)
y = telescope['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=3, weights='distance')
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"Test Accuracy: {knn.score(X_test, y_test)}")
print(f"Train Accuracy:{knn.score(X_train, y_train)}")
print(f"Difference in Accuracy: {knn.score(X_test, y_test) - knn.score(X_train, y_train)}")

#### Discussion
Comparison and discuss the differences you see with distance weighting and normalization vs without.

when we normalized and used distance weighting, we see an increase in the training accuracy (up from 98.8% and 99.9%) and a higher accuracy than when we just used normalizing. we also see a greater difference between the training and testing accuracy than when we just normalize the data.

### 2.4 (10%) Different k Values
- Using your normalized data with distance weighting, create one graph with classification accuracy on the test set on the y-axis and k values on the x-axis.
- Use values of k from 1 to 15.  Use the same train/test split for each. 

In [ ]:
# Calculate and Graph classification accuracy vs k values
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Split features and target
X = telescope.drop('class', axis=1)
y = telescope['class']

# Use a fixed 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Store accuracies for different k values
k_values = range(1, 16)
test_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn.fit(X_train, y_train)
    test_accuracies.append(knn.score(X_test, y_test))

# Plot
plt.figure(figsize=(8,5))
plt.plot(k_values, test_accuracies, marker='o')
plt.xticks(k_values)
plt.xlabel("k (number of neighbors)")
plt.ylabel("Test Set Accuracy")
plt.title("KNN Accuracy vs. k (Normalized Data + Distance Weighting)")
plt.grid(True)
plt.show()


#### Discussion
How do the k values affect your results?

Looking at the graph, we can see that there seems to be a "sweetspot" telling us that we get the most accuracy when we have 8 as our k values, this leads to a much higher accuracy than other values of k. We also see a drop in accuracy with higher values of k.

## 3 KNN Regression with normalization and distance weighting

Use the [sklean KNeighborsRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html#sklearn.neighbors.KNeighborsRegressor) on the [housing price prediction](https://axon.cs.byu.edu/data/uci_regression/housing.arff) problem.  
### 3.1 (5%) Ethical Data
Note this data set has an example of an inappropriate input feature which we discussed.  State which feature is inappropriate and discuss why.

#### Discussion
Discuss the innapropriate feature. Which one and why?

In this case, the inappropriate feature would be "B". This feature would be inappropriate because it encodes race and when we use a model, it will be used to predict housing prices which could reinforce social discriminitory patterns.

### 3.2 (15%) - KNN Regression 
- Do random 80/20 train/test splits each time
- Run with k=3
- Print the score (coefficient of determination) and Mean Absolute Error (MAE) for the train and test set for the cases of
  - No input normalization and no distance weighting
  - Normalization and no distance weighting
  - Normalization and distance weighting
- Normalize inputs features where needed but do not normalize the output

In [ ]:
# Learn and experiment with housing price prediction data
housing = pd.read_csv("HousingData.csv")
housing.head()
housing.isna()

In [ ]:
housing

In [ ]:
#no normalization and no distance weighting
housing.dropna(inplace=True)

knn = KNeighborsRegressor(n_neighbors=3, weights='uniform')
X = housing.drop('MEDV', axis=1)
y = housing['MEDV']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"Test Score: {knn.score(X_test, y_test)}")
print(f"Train Score:{knn.score(X_train, y_train)}")
print(f"Test MAE: {np.mean(np.abs(y_test - y_pred))}")
print(f"Train MAE: {np.mean(np.abs(y_train - knn.predict(X_train)))}")

In [ ]:
#with normalization and no distance weighting
housing.dropna(inplace=True)
knn = KNeighborsRegressor(n_neighbors=3, weights='uniform')
X = housing.drop('MEDV', axis=1)
y = housing['MEDV']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"Test Score: {knn.score(X_test, y_test)}")
print(f"Train Score:{knn.score(X_train, y_train)}")
print(f"Test MAE: {np.mean(np.abs(y_test - y_pred))}")
print(f"Train MAE: {np.mean(np.abs(y_train - knn.predict(X_train)))}")

In [ ]:
#with normalization and distance weighting
housing.dropna(inplace=True)
knn = KNeighborsRegressor(n_neighbors=3, weights='distance')
X = housing.drop('MEDV', axis=1)
y = housing['MEDV']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
print(f"Test Score: {knn.score(X_test, y_test)}")
print(f"Train Score:{knn.score(X_train, y_train)}")
print(f"Test MAE: {np.mean(np.abs(y_test - y_pred))}")
print(f"Train MAE: {np.mean(np.abs(y_train - knn.predict(X_train)))}")

#### Discussion
Discuss your results. How did the hyperparameters affect your results? Discuss each one and combinations of each.

When doing our model with no normalization and no distance weighting, we get the highest MAE score indicating that out of the three models that we have, this model performs the worst. Our model with normalizing but not distance wieghing only slightly did better than when we introduced the distance weighing. It is interesting, however, that our model with normalization and distance weighting performed the best on the training set

### 3.3 (10%)  Different k Values
- Using housing with normalized data and distance weighting, create one graph with MAE on the test set on the y-axis and k values on the x-axis
- Use values of k from 1 to 15.  Use the same train/test split for each. 

In [ ]:
# Learn and graph for different k values

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor

# Features and target
X = housing.drop('MEDV', axis=1)
y = housing['MEDV']

# 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Impute missing values with column mean
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Store test MAE for different k values
k_values = range(1, 16)
test_mae = []

for k in k_values:
    knn = KNeighborsRegressor(n_neighbors=k, weights='distance')
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    test_mae.append(np.mean(np.abs(y_test - y_pred)))


best_k = k_values[np.argmin(test_mae)]
best_mae = min(test_mae)

print(f"Best k value: {best_k} with MAE: {best_mae:.4f}")

# Plot
plt.figure(figsize=(8,5))
plt.plot(k_values, test_mae, marker='o')
plt.xticks(k_values)
plt.xlabel("k (number of neighbors)")
plt.ylabel("Test MAE")
plt.title("KNN Regression: Test MAE vs. k (Normalized + Distance Weighting)")
plt.grid(True)
plt.show()



#### Discussion
How did the k values affect your results for this dataset? How does that compare to your previous work in this lab?

From the graph, it appears that when we have k=3 and from there it seems to go up indicating that there is more variability between our predictions and the true value when we take into account greater k values. When we used k=3, we found the MAE to be 2.9521. It is interesting that as we take into account more neighbours with this dataset, out MAE gets higher indicating that the model becomes worse.

## 4. (20%) KNN with nominal and real data

- Use the [lymph dataset](https://axon.cs.byu.edu/data/uci_class/lymph.arff)
- Use a 80/20 split of the data for the training/test set
- This dataset has both continuous and nominal attributes 
- Implement a distance metric which uses Euclidean distance for continuous features and 0/1 distance for nominal. Hints:
    - Write your own distance function (e.g. mydist) and use clf = KNeighborsClassifier(metric=mydist)
    - Change the nominal features in the data set to integer values since KNeighborsClassifier expects numeric features. I used Label_Encoder on the nominal features.
    - Keep a list of which features are nominal which mydist can use to decide which distance measure to use
    - There was an occasional bug in SK version 1.3.0 ("Flags object has no attribute 'c_contiguous'") that went away when I upgraded to the lastest SK version 1.3.1 
- Use your own choice for k and other parameters

In [ ]:
column_names = ['mcv', 'alkphos', 'sgpt', 'sgot', 'gammagt', 'drinks', 'selector']
lymph = pd.read_csv("bupa.data",names = column_names)
lymph.describe()

In [ ]:
X = lymph.drop('selector', axis=1)
y = lymph['selector']

In [ ]:
def custom_weighted_distance(x1, x2):
    # Higher weight = more important
    #Each one corresponds with a feature, if you want to see the features, look at bupa.names. I also
    #just followed the column name order from above
    weights = np.array([
        0.8,
        0.9,
        1.3,
        1.3,
        1.5,
        0.7
    ])

    diff = x1 - x2
    weighted_diff_squared = weights * (diff ** 2)
    distance = np.sqrt(np.sum(weighted_diff_squared))

    return distance

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}\n")

# Normalize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# 1. Standard Euclidean
knn_euclidean = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn_euclidean.fit(X_train_scaled, y_train)
train_acc_euc = knn_euclidean.score(X_train_scaled, y_train)
test_acc_euc = knn_euclidean.score(X_test_scaled, y_test)
print(f"\n1. Standard Euclidean Distance (Baseline):")
print(f"   Train Accuracy: {train_acc_euc:.4f}")
print(f"   Test Accuracy:  {test_acc_euc:.4f}")

# 2.Custom Euclidean
knn_weighted = KNeighborsClassifier(n_neighbors=5, metric=custom_weighted_distance)
knn_weighted.fit(X_train_scaled, y_train)
train_acc_weighted = knn_weighted.score(X_train_scaled, y_train)
test_acc_weighted = knn_weighted.score(X_test_scaled, y_test)
print(f"\n2. Custom Weighted Euclidean Distance:")
print(f"   Train Accuracy: {train_acc_weighted:.4f}")
print(f"   Test Accuracy:  {test_acc_weighted:.4f}")

# 3. distance weighting
knn_dist_weighted = KNeighborsClassifier(
    n_neighbors=5,
    metric=custom_weighted_distance,
    weights='distance'
)
knn_dist_weighted.fit(X_train_scaled, y_train)
train_acc_dw = knn_dist_weighted.score(X_train_scaled, y_train)
test_acc_dw = knn_dist_weighted.score(X_test_scaled, y_test)
print(f"\n3. Custom Weighted Euclidean + Distance Weighting:")
print(f"   Train Accuracy: {train_acc_dw:.4f}")
print(f"   Test Accuracy:  {test_acc_dw:.4f}")

#k values
print("Testing Different k Values (Custom Weighted + Distance Weighting):")

k_values = range(1, 16)
test_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric=custom_weighted_distance,
        weights='distance'
    )
    knn.fit(X_train_scaled, y_train)
    test_acc = knn.score(X_test_scaled, y_test)
    test_accuracies.append(test_acc)
    print(f"k = {k:2d}: Test Accuracy = {test_acc:.4f}")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(k_values, test_accuracies, marker='o', linewidth=2, markersize=8)
plt.xlabel('k (number of neighbors)', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('KNN with Custom Weighted Distance Metric\n(BUPA Liver Disorders Dataset)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(k_values)
plt.axhline(y=test_acc_euc, color='r', linestyle='--', alpha=0.5, label='Baseline (Euclidean)')
plt.legend()
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(test_accuracies)]
best_acc = max(test_accuracies)
print(f"\nBest k value: {best_k} with accuracy: {best_acc:.4f}")
print(f"Improvement over baseline: {best_acc - test_acc_euc:.4f}")

#### Discussion
Explain your distance metric and discuss your results

I assigned different weights to the different features depending on how I determined the importance of the feature, a more important feature got a higher weight. With this new distance metric I was able to get slightly better accuracy than using the standard euclidiean distance metric. In the graph you can see that by using the regular Euclidean distance we got an accuracy of .71. I believe that with further tuning of the weights, this new distance metric could improve the accuracy. We would just need more data and possibly a domain expert to better understand the different features that we have. That way we can have more accuracy for the importance of a feature, not just me choosing numbers from deciding the importance of a feature.

## 5. (Optional 15% extra credit) Code up your own KNN Learner 
Below is a scaffold you could use if you want. Requirements for this task:
- Your model should support the methods shown in the example scaffold below
- Use Euclidean distance to decide closest neighbors
- Implement both the classification and regression versions
- Include optional distance weighting for both algorithms
- Run your algorithm on the magic telescope and housing data sets above and discuss and compare your results 

*Discussion*

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

class KNNClassifier(BaseEstimator,ClassifierMixin):
    def __init__(self, columntype=[], weight_type='inverse_distance'): ## add parameters here
        """
        Args:
            columntype for each column tells you if continues[real] or if nominal[categoritcal].
            weight_type: inverse_distance voting or if non distance weighting. Options = ["no_weight","inverse_distance"]
        """
        self.columntype = columntype #Note This won't be needed until part 5
        self.weight_type = weight_type

    def fit(self, data, labels):
        """ Fit the data; run the algorithm (for this lab really just saves the data :D)
        Args:
            X (array-like): A 2D numpy array with the training data, excluding targets
            y (array-like): A 2D numpy array with the training targets
        Returns:
            self: this allows this to be chained, e.g. model.fit(X,y).predict(X_test)
        """
        return self
    
    def predict(self, data):
        """ Predict all classes for a dataset X
        Args:
            X (array-like): A 2D numpy array with the training data, excluding targets
        Returns:
            array, shape (n_samples,)
                Predicted target values per element in X.
        """
        pass

    #Returns the Mean score given input data and labels
    def score(self, X, y):
        """ Return accuracy of model on a given dataset. Must implement own score function.
        Args:
            X (array-like): A 2D numpy array with data, excluding targets
            y (array-like): A 2D numpy array with targets
        Returns:
            score : float
                Mean accuracy of self.predict(X) wrt. y.
        """
        return 0